# BraTS-METs Evaluation Example

This notebook provides a step-by-step example of how to use the `brats-evaluate` and `brats-parse-metrics` console scripts (installed by the `BraTS-evaluation` package) to evaluate brain tumor segmentations.

The sample data used here is from the training set of BraTS 2025 Brain Metastasis (METS) challenge. It includes four cases, each with a reference (ground truth) and a corresponding prediction. While this example
focuses on METS, the same evaluation pipeline can be adapted for other segmentation tasks like glioma, meningioma, or pediatric tumors by choosing the appropriate `--config` name (e.g., `gli`, `ped`, `MenRT`, `MenPre`, `GoAT`).

## Sample Data Structure
The data is organized as follows (pay attention to the naming convention for the predicted masks required by the BraTS challege ):

```
├── pred
│   ├── 00132-000.nii.gz
│   ├── BraTS-MET-00630-001.nii.gz
│   ├── BraTS-MET_01325-000.nii.gz
│   └── SEG-00217-000.nii.gz
└── ref
    ├── BraTS-MET-00132-000-seg.nii.gz
    ├── BraTS-MET-00217-000-seg.nii.gz
    ├── BraTS-MET-00630-001-seg.nii.gz
    └── BraTS-MET-01325-000-seg.nii.gz


```


### 1. Run the Evaluation

First, we call `brats-evaluate` to compare the predictions against the references. This command will generate a detailed JSON file containing all the raw metrics from Panoptica.

In [ ]:
!brats-evaluate \
    --config mets \
    --ref_path ./sample_data/ref/ \
    --pred_path ./sample_data/pred/ \
    --summary_json ./sample_mets_metrics.json

### 2. Inspect the JSON Output

The generated JSON file is highly detailed. Let's load it and print the metrics for the first subject to see the rich, instance-wise information Panoptica provides.

In [2]:
import json

with open('sample_mets_metrics.json', 'r') as f:
    data = json.load(f)
# Pretty-print the first subject's metrics,
print(json.dumps(data['metrics'][0], indent=4))

{
    "et": {
        "n_ref_instances": 8,
        "n_pred_instances": 7,
        "tp": 6,
        "fp": 1,
        "fn": 2,
        "prec": 0.8571428571428571,
        "rec": 0.75,
        "rq": 0.8,
        "sq_dsc": 0.7735988324659234,
        "sq_dsc_std": 0.05146092409268618,
        "pq_dsc": 0.6188790659727388,
        "sq_hd95": 2.0748032315081146,
        "sq_hd95_std": 1.0356253343863966,
        "sq_nsd": 0.8605908094646745,
        "sq_nsd_std": 0.15545837198420034,
        "instance_voxel_count_ref": 916.1666666666666,
        "instance_volume_ref": 916.1666666666666,
        "global_bin_volume_pred": 4800,
        "global_bin_volume_ref": 6915,
        "global_bin_dsc": 0.6840802390098165,
        "global_bin_hd95": 33.52610922848042,
        "global_bin_nsd": 0.32470764194847074,
        "reference_instances": [
            {
                "sq_dsc": 0.7091442282058118,
                "sq_hd95": 3.1622776601683795,
                "sq_nsd": 0.5571437139665475,
       

### 3. Parse the Results into a CSV

Next, we use the `brats-parse-metrics` command with the `mets` subcommand to process the raw JSON file.
This script calculates size-stratified statistics and organizes the results into a clean, human-readable CSV file.

In [ ]:
!brats-parse-metrics mets \
    --json_path ./sample_mets_metrics.json \
    --output_csv_path ./sample_mets_summary.csv

### 4. Display the Final CSV Report

Finally, let's load and display the generated CSV file. This table provides a clear summary of the evaluation, making it easy to compare performance across different subjects and metrics.

In [4]:
import pandas as pd
df = pd.read_csv('sample_mets_summary.csv')
display(df)

,subject_id,all_instance_tp_et,all_instance_fp_et,all_instance_fn_et,all_instance_f1_et,lesionwise_dsc_mean_et,lesionwise_dsc_std_et,lesionwise_hd95_mean_et,lesionwise_hd95_std_et,lesionwise_nsd_mean_et,...,lesionwise_dsc_mean_wt,lesionwise_dsc_std_wt,lesionwise_hd95_mean_wt,lesionwise_hd95_std_wt,lesionwise_nsd_mean_wt,lesionwise_nsd_std_wt,small_instance_tp_wt,small_instance_fn_wt,small_instance_fp_wt,small_instance_f1_wt
0,BraTS-MET-00132-000-seg.nii.gz,6.000000,1.000000,2.000000,0.800000,0.773599,0.051461,2.074803,1.035625,0.860591,...,0.691078,0.163653,2.658782,1.214444,0.775203,0.142018,NaN,NaN,NaN,NaN
1,BraTS-MET-00217-000-seg.nii.gz,27.000000,7.000000,7.000000,0.794118,0.727639,0.151173,1.672625,0.736172,0.897355,...,0.635117,0.210657,5.826932,7.705425,0.706321,0.184169,1.0,3.00000,9.000000,0.142857
2,BraTS-MET-00630-001-seg.nii.gz,12.000000,0.000000,0.000000,1.000000,0.876783,0.037458,0.898200,0.106155,0.970356,...,0.858429,0.060251,1.182275,0.400099,0.933376,0.057534,1.0,0.00000,0.000000,1.000000
3,BraTS-MET-01325-000-seg.nii.gz,1.000000,0.000000,1.000000,0.666667,0.973133,0.000000,0.500000,0.000000,0.953838,...,0.985481,0.000000,0.500000,0.000000,0.963291,0.000000,NaN,NaN,NaN,NaN
4,mean,11.500000,2.000000,2.500000,0.815196,0.837789,0.060023,1.286407,0.469488,0.920535,...,0.792526,0.108640,2.541997,2.329992,0.844547,0.095930,1.0,1.50000,4.500000,0.571429
5,std,11.269428,3.366502,3.109126,0.137706,0.109685,0.064533,0.716443,0.498016,0.050734,...,0.159837,0.095879,2.368053,3.619073,0.123701,0.082839,0.0,2.12132,6.363961,0.606092


As shown above, the last two rows of the DataFrame conveniently provide the **mean** and **standard deviation (std)** for each metric column, calculated over all valid (non-NaN) rows. This gives a quick statistical overview of the algorithm's performance across the entire dataset."